In [1]:
!pip install -q transformers torch accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.0 MB/s eta 0:00:00


In [2]:
!hf auth login

Hint: A new version of huggingface_hub (1.26.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: HD2R-2AC1

    Waiting for authorization...
Token is valid.
The token `oauth-afsdfsadfasf` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-afsdfsadfasf`
Note: This token will be refreshed automatically when it expires.


In [63]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Initializing {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

Initializing meta-llama/Llama-3.2-3B-Instruct...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [64]:
if hasattr(model.model, "_forward_hooks"):
    model.model._forward_hooks.clear()

for layer in model.model.layers:
    if hasattr(layer, "_forward_hooks"):
        layer._forward_hooks.clear()

print("all stale hooks flushed successfully! VRAM is clean.")

def sparsify_activations(hidden_states, keep_ratio=1.0):

    magnitudes = torch.abs(hidden_states)

    channel_dim = hidden_states.size(-1)
    k = int(channel_dim * keep_ratio)

    topk_values, _ = torch.topk(magnitudes, k, dim=-1)

    thresholds = topk_values[..., -1].unsqueeze(-1)

    mask = magnitudes >= thresholds

    return hidden_states * mask
def calculate_topk_entropy(logits, k=10):
    last_token_logits = logits[0, -1, :]

    topk_logits, _ = torch.topk(last_token_logits, k, dim=-1)

    probs = torch.softmax(topk_logits, dim=-1)

    entropy = -torch.sum(probs * torch.log2(probs + 1e-9), dim=-1)

    return entropy.item()
def make_hook(layer_idx):

    def hook(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
            processed_states = sparsify_activations(hidden_states, keep_ratio=0.35)
            return (processed_states,) + output[1:]

        return sparsify_activations(output, keep_ratio=0.35)
    return hook
num_layers = len(model.model.layers)
start_layer = num_layers // 4
end_layer = (3 * num_layers) // 4

print(f"injecting Sparsification Hooks into layers {start_layer} through {end_layer}...")
for i in range(start_layer, end_layer):
    model.model.layers[i].register_forward_hook(make_hook(i))


all stale hooks flushed successfully! VRAM is clean.
injecting Sparsification Hooks into layers 7 through 21...


In [65]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)




prompt = "whats your opinion on the future of decentralized computing architecture"
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

print(f"Prompt: \"{prompt}\"\n" + "-" * 60)

input_ids = inputs["input_ids"]
attention_mask = inputs.get("attention_mask", None)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1000,        # Give the 3B brain real room to answer!
        do_sample=True,

        # The Optimized Llama-3 Sampler Stack
        temperature=0.7,           # Lower temperature for much tighter logical focus
        top_p=0.9,                 # Restrict generation to the top 90% probability mass
        repetition_penalty=1.05    # A tiny, gentle nudge instead of a sledgehammer
    )


generated_text = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"Engine Output:\n{generated_text}")
print("-" * 60)




Prompt: "whats your opinion on the future of decentralized computing architecture"
------------------------------------------------------------
Engine Output:
The future of decentralized computing architecture is a rapidly evolving field that has been shaped by the integration of emerging technologies, innovative applications, and the integration of various data and decentralized systems.

**Key Aspects**

1. **Decentralization**: The decentralized architecture offers a more efficient and flexible way to manage various data storage and processing needs.
2. **Blockchain Technology**: Decentralized systems utilize blockchain technology to optimize resource allocation and the decentralized architecture of various decentralized networks.
3 **Artificial Intelligence and IoT**: Artificial intelligence and IoT enable decentralized architecture with an interconnected network of decentralized decentralized systems.

**Advantages**

1. **Efficient Resource Optimization**: The decentralized archi

In [62]:
model = 0
